In [68]:
import sqlite3
import pandas as pd
import yfinance as yf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [69]:
# =====================================================================
# NumPy 手寫 Logistic Regression 核心演算法 (NumPy 矩陣運算)
# =====================================================================
class MyLogisticRegression:
    def __init__(self, learning_rate=0.01, iterations=1000):
        self.lr = learning_rate
        self.iterations = iterations
        self.weights = None
        self.bias = None

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)
        num_samples, num_features = X.shape

        self.weights = np.zeros((num_features, 1))
        self.bias = 0.0

        for i in range(self.iterations):
            linear_model = np.dot(X, self.weights) + self.bias
            predictions = self._sigmoid(linear_model)

            dw = (1 / num_samples) * np.dot(X.T, (predictions - y))
            db = (1 / num_samples) * np.sum(predictions - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict_proba(self, X):
        X = np.array(X)
        linear_model = np.dot(X, self.weights) + self.bias
        return self._sigmoid(linear_model)

    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)
        return (probabilities >= threshold).astype(int).flatten()

In [70]:
def run_end_to_end_pipeline(target_ticker="2330.TW", start_date="2024-01-01", end_date="2026-07-01", db_name="stock_data_v2.db"):
    # -----------------------------------------------------------------
    # 數據下載
    # -----------------------------------------------------------------
    print("正在下載數據...")
    df_target = yf.download(target_ticker, start=start_date, end=end_date)
    df_sox = yf.download("^SOX", start=start_date, end=end_date)
    df_gspc = yf.download("^GSPC", start=start_date, end=end_date)

    if isinstance(df_target.columns, pd.MultiIndex):
        df_target.columns = df_target.columns.get_level_values(0)
    if isinstance(df_sox.columns, pd.MultiIndex):
        df_sox.columns = df_sox.columns.get_level_values(0)
    if isinstance(df_gspc.columns, pd.MultiIndex):
        df_gspc.columns = df_gspc.columns.get_level_values(0)

    df_target.index.name = "Date"
    df_sox.index.name = "Date"
    df_gspc.index.name = "Date"
    
    df_target = df_target[["Open", "High", "Low", "Close", "Volume"]]
    df_sox = df_sox[["Open", "High", "Low", "Close", "Volume"]]
    df_gspc = df_gspc[["Open", "High", "Low", "Close", "Volume"]]

    # -----------------------------------------------------------------
    # 資料清洗
    # -----------------------------------------------------------------
    print("\n" + "="*60)
    print(" 啟動資料清洗")
    print("="*60)
    
    # 1. 去除重複值
    df_t = df_target[~df_target.index.duplicated(keep='first')].copy()
    df_s = df_sox[~df_sox.index.duplicated(keep='first')].copy()
    df_g = df_gspc[~df_gspc.index.duplicated(keep='first')].copy()
    
    # 2. 價格一致性物理檢驗 (High >= Low 等物理限制)
    for df_name, df_temp in [("台股", df_t), ("費半", df_s), ("標普", df_g)]:
        before_count = len(df_temp)
        valid_mask = (
            (df_temp['High'] >= df_temp['Low']) &
            (df_temp['High'] >= df_temp['Open']) &
            (df_temp['High'] >= df_temp['Close']) &
            (df_temp['Low'] <= df_temp['Open']) &
            (df_temp['Low'] <= df_temp['Close']) &
            (df_temp['Volume'] >= 0)
        )
        if df_name == "台股":
            df_t = df_temp[valid_mask].copy()
        elif df_name == "費半":
            df_s = df_temp[valid_mask].copy()
        elif df_name == "標普":
            df_g = df_temp[valid_mask].copy()
            
        after_count = len(df_temp[valid_mask])
        if before_count != after_count:
            print(f"[Warning] {df_name} 偵測到 {before_count - after_count} 筆價格邏輯錯亂的髒數據，已直接剔除。")

    # 3. 台灣漲跌幅限制物理過濾 (10% Limit)
    df_t["TW_Ret_1D"] = df_t["Close"].pct_change(1)
    outlier_mask = df_t["TW_Ret_1D"].abs() > 0.105
    outliers_detected = outlier_mask.sum()
    if outliers_detected > 0:
        print(f"[Warning] 偵測到 {outliers_detected} 筆超出台股 10% 限制的異常報酬，已啟動平滑化修正。")
        df_t.loc[outlier_mask, ["Open", "High", "Low", "Close", "Volume", "TW_Ret_1D"]] = None
        df_t.ffill(inplace=True)

    # -----------------------------------------------------------------
    # 特徵工程、時序對齊與寫入資料庫
    # -----------------------------------------------------------------
    
    # 1. 計算台股技術指標
    df_t["TW_Ret_1D"] = df_t["Close"].pct_change(1)
    df_t["TW_Ret_5D"] = df_t["Close"].pct_change(5)

    delta = df_t["Close"].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df_t["TW_RSI_14"] = 100 - (100 / (1 + rs))

    ma20 = df_t["Close"].rolling(window=20).mean()
    std20 = df_t["Close"].rolling(window=20).std()
    df_t["TW_BB_Width"] = (4 * std20) / ma20
    df_t["TW_Vol_Ratio_5D"] = df_t["Volume"] / df_t["Volume"].rolling(window=5).mean()

    # 2. 計算美股特徵，並執行防洩漏時序平移 (Shift 1)
    df_sox_feats = pd.DataFrame(index=df_s.index)
    df_sox_feats["US_SOX_Ret_1D"] = df_s["Close"].pct_change(1)
    df_sox_feats["US_SOX_Ret_5D"] = df_s["Close"].pct_change(5)

    df_gspc_feats = pd.DataFrame(index=df_g.index)
    df_gspc_feats["US_GSPC_Ret_1D"] = df_g["Close"].pct_change(1)
    df_gspc_feats["US_GSPC_Ret_5D"] = df_g["Close"].pct_change(5)

    # 用前一天的美股數據 (T-1) 預測今天的台股 (T)
    df_sox_feats = df_sox_feats.shift(1)
    df_gspc_feats = df_gspc_feats.shift(1)

    # 跨市場合併
    df_merged = df_t.join(df_sox_feats, how="left").ffill()
    df_merged = df_merged.join(df_gspc_feats, how="left").ffill()
    df_merged.dropna(inplace=True)

    # 寫入 SQL 資料庫
    conn = sqlite3.connect(db_name)
    df_to_save = df_merged.reset_index()
    df_to_save["Date"] = pd.to_datetime(df_to_save["Date"]).dt.strftime("%Y-%m-%d")
    df_to_save.to_sql("stock_features_v2", conn, if_exists="replace", index=False)
    conn.close()
    print("資料庫寫入成功。")
    
    return df_merged

In [71]:
# =====================================================================
# 第二段：相關係數與熱點圖分析
# =====================================================================
def analyze_correlations(df_cleaned, feature_cols):
    print("\n" + "="*50)
    print("特徵相關係數與熱點圖分析")
    print("="*50)
    df_analysis = df_cleaned.copy()
    df_analysis['Next_Close'] = df_analysis['Close'].shift(-1)
    df_analysis['Target'] = (df_analysis['Next_Close'] > df_analysis['Close']).astype(int)
    df_analysis.dropna(inplace=True)
    
    all_cols = feature_cols + ['Target']
    pearson_corr = df_analysis[all_cols].corr(method='pearson')
    spearman_corr = df_analysis[all_cols].corr(method='spearman')
    
    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, linewidths=0.5)
    plt.title("Pearson Correlation Heatmap (Linear)", fontsize=12, fontweight='bold')
    
    plt.subplot(1, 2, 2)
    sns.heatmap(spearman_corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, linewidths=0.5)
    plt.title("Spearman Correlation Heatmap (Rank/Trend)", fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig("correlation_heatmap_v2.png", dpi=300)
    plt.close()
    print("[Plot] 熱點圖已成功儲存為 'correlation_heatmap_v2.png'")
    
    print("\n[Ranking] 各特徵與明日漲跌(Target)的關聯度排名：")
    pearson_rank = pearson_corr['Target'].drop('Target').abs().sort_values(ascending=False)
    spearman_rank = spearman_corr['Target'].drop('Target').abs().sort_values(ascending=False)
    
    print("-" * 65)
    print(f"{'排名':^4} | {'Pearson (線性關聯)':^25} | {'Spearman (趨勢關聯)':^25}")
    print("-" * 65)
    for i in range(len(feature_cols)):
        p_feat = pearson_rank.index[i]
        p_val = pearson_corr['Target'][p_feat]
        s_feat = spearman_rank.index[i]
        s_val = spearman_corr['Target'][s_feat]
        print(f" #{i+1:2d} | {p_feat[:15]:15s} ({p_val:+.4f}) | {s_feat[:15]:15s} ({s_val:+.4f})")
    print("-" * 65)
    
    return df_analysis

In [72]:
# =====================================================================
# 隨機森林模型
# =====================================================================
def run_random_forest(X_train, X_test, y_train, y_test, feature_cols, window_size=30):
    print("\n" + "="*50)
    print("隨機森林")
    print("="*50)
    
    X_combined = np.vstack([X_train.values, X_test.values])
    y_combined = np.concatenate([y_train.values, y_test.values])
    
    test_start_idx = len(X_train)
    total_len = len(X_combined)
    
    rf_preds = []
    rf_actuals = []
    
    for t in range(test_start_idx, total_len):
        X_train_window = X_combined[t - window_size : t]
        y_train_window = y_combined[t - window_size : t]
        
        X_test_today = X_combined[t].reshape(1, -1)
        y_test_today = y_combined[t]
        
        # 局部標準化防止資料洩漏 (Data Leakage)
        mean = X_train_window.mean(axis=0)
        std = X_train_window.std(axis=0) + 1e-15
        
        X_train_scaled = (X_train_window - mean) / std
        X_test_scaled = (X_test_today - mean) / std
        
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_train_scaled, y_train_window)
        
        pred_today = rf.predict(X_test_scaled)[0]
        
        rf_preds.append(pred_today)
        rf_actuals.append(y_test_today)
        
    rf_preds = np.array(rf_preds)
    rf_actuals = np.array(rf_actuals)
    rf_acc = accuracy_score(rf_actuals, rf_preds)
    
    print(f"隨機森林預測準確度 (Accuracy): {rf_acc:.4f}")
    
    importances = rf.feature_importances_
    indices = np.argsort(importances)[::-1]
    print("\n隨機森林特徵重要度排序：")
    for f in range(len(feature_cols)):
        print(f" #{f+1:2d} | {feature_cols[indices[f]]:20s} : {importances[indices[f]]:.4f}")
        
    return rf_preds, rf_acc

In [73]:
# =====================================================================
# XGBoost 模型
# =====================================================================
def run_xgboost(X_train, X_test, y_train, y_test, feature_cols, window_size=30):
    print("\n" + "="*50)
    print("XGBoost")
    print("="*50)
    
    X_combined = np.vstack([X_train.values, X_test.values])
    y_combined = np.concatenate([y_train.values, y_test.values])
    
    test_start_idx = len(X_train)
    total_len = len(X_combined)
    
    xgb_preds = []
    xgb_actuals = []
    
    for t in range(test_start_idx, total_len):
        X_train_window = X_combined[t - window_size : t]
        y_train_window = y_combined[t - window_size : t]
        
        X_test_today = X_combined[t].reshape(1, -1)
        y_test_today = y_combined[t]
        
        mean = X_train_window.mean(axis=0)
        std = X_train_window.std(axis=0) + 1e-15
        
        X_train_scaled = (X_train_window - mean) / std
        X_test_scaled = (X_test_today - mean) / std
        
        xgb = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, eval_metric='logloss')
        xgb.fit(X_train_scaled, y_train_window)
        
        pred_today = xgb.predict(X_test_scaled)[0]
        
        xgb_preds.append(pred_today)
        xgb_actuals.append(y_test_today)
        
    xgb_preds = np.array(xgb_preds)
    xgb_actuals = np.array(xgb_actuals)
    xgb_acc = accuracy_score(xgb_actuals, xgb_preds)
    
    print(f"XGBoost預測準確度 (Accuracy): {xgb_acc:.4f}")
    
    importances = xgb.feature_importances_
    indices = np.argsort(importances)[::-1]
    print("\nXGBoost特徵重要度排序：")
    for f in range(len(feature_cols)):
        print(f" #{f+1:2d} | {feature_cols[indices[f]]:20s} : {importances[indices[f]]:.4f}")
        
    return xgb_preds, xgb_acc

In [74]:
# =====================================================================
# 自製模型
# =====================================================================
def run_my_logistic_regression(X_train, X_test, y_train, y_test, feature_cols, window_size=30):
    print("\n" + "="*50)
    print("自製回歸模型")
    print("="*50)
    
    X_combined = np.vstack([X_train.values, X_test.values])
    y_combined = np.concatenate([y_train.values, y_test.values])
    
    test_start_idx = len(X_train)
    total_len = len(X_combined)
    
    my_preds = []
    my_actuals = []
    
    for t in range(test_start_idx, total_len):
        X_train_window = X_combined[t - window_size : t]
        y_train_window = y_combined[t - window_size : t]
        
        X_test_today = X_combined[t].reshape(1, -1)
        y_test_today = y_combined[t]
        
        mean = X_train_window.mean(axis=0)
        std = X_train_window.std(axis=0) + 1e-15
        
        X_train_scaled = (X_train_window - mean) / std
        X_test_scaled = (X_test_today - mean) / std
        
        my_model = MyLogisticRegression(learning_rate=0.05, iterations=200)
        my_model.fit(X_train_scaled, y_train_window)
        
        pred_today = my_model.predict(X_test_scaled)[0]
        
        my_preds.append(pred_today)
        my_actuals.append(y_test_today)
        
    my_preds = np.array(my_preds)
    my_actuals = np.array(my_actuals)
    my_acc = accuracy_score(my_actuals, my_preds)
    
    print(f"自製模型的預測準確度 (Accuracy): {my_acc:.4f}")
    
    print("\n自製模型的特徵權重排序：")
    weights_flat = my_model.weights.flatten()
    indices = np.argsort(np.abs(weights_flat))[::-1]
    
    for f in range(len(feature_cols)):
        col_idx = indices[f]
        print(f" #{f+1:2d} | {feature_cols[col_idx]:20s} : {weights_flat[col_idx]:+.4f}")
        
    return my_preds, my_acc

In [80]:
def compare_models(rf_acc, xgb_acc, my_acc):
    print("\n" + "="*50)
    print("三模型預測效能比較")
    print("="*50)
    print(f"隨機森林 (Random Forest) 準確度: {rf_acc:.4f}")
    print(f"XGBoost 模型 準確度:            {xgb_acc:.4f}")
    print(f"手寫 Logistic 模型的 準確度:    {my_acc:.4f}")
    
    results = {
        "隨機森林": rf_acc,
        "XGBoost": xgb_acc,
        "手寫 Logistic": my_acc
    }
    winner = max(results, key=results.get)
    sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
    
    print("-" * 50)
    print(f"預測勝出模型: {winner}")
    if sorted_results[0][1] > sorted_results[1][1]:
        diff = sorted_results[0][1] - sorted_results[1][1]
        print(f"勝出準確度差距: {diff:.4f}")
    print("="*50)

In [81]:
if __name__ == "__main__":
    db_file = "stock_data_v2.db"
    
    feature_cols = [
        'Open', 'High', 'Low', 'Close', 'Volume', 
        'TW_Ret_1D', 'TW_Ret_5D', 'TW_RSI_14', 'TW_BB_Width', 'TW_Vol_Ratio_5D',
        'US_SOX_Ret_1D', 'US_SOX_Ret_5D', 'US_GSPC_Ret_1D', 'US_GSPC_Ret_5D'
    ]
    
    # 0. 下載、清洗與儲存至資料庫
    df_raw_merged = run_end_to_end_pipeline(target_ticker="2330.TW", db_name=db_file)
    
    # 建立模型預測目標
    df_model = df_raw_merged.copy()
    df_model['Next_Close'] = df_model['Close'].shift(-1)
    df_model['Target'] = (df_model['Next_Close'] > df_model['Close']).astype(int)
    df_model.dropna(inplace=True)
    
    # 1. 相關性分析
    df_model = analyze_correlations(df_model, feature_cols)
    
    # 2. 切分時序資料集
    X = df_model[feature_cols]
    y = df_model['Target']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)
    
    # 定義我們要測試的所有滑動窗口天數
    window_sizes = [30, 60, 90, 120, 150, 180, 210, 240, 270, 310, 340, 370]
    
    # 用於儲存最終結果的字典
    results_summary = []
    
    # 3. 滾動執行不同窗口大小的基準測試 (Benchmarking)
    for ws in window_sizes:
        print("\n" + "="*70)
        print(f"[系統測試] 開始評估滑動窗口天數: {ws} 天")
        print("="*70)
        
        # 靜默或正常運行隨機森林 (傳入當前 window_size)
        _, rf_acc = run_random_forest(X_train, X_test, y_train, y_test, feature_cols, window_size=ws)
        
        # 滾動運行 XGBoost (傳入當前 window_size)
        _, xgb_acc = run_xgboost(X_train, X_test, y_train, y_test, feature_cols, window_size=ws)
        
        # 滾動運行自製手寫 Logistic 回歸 (傳入當前 window_size)
        _, my_acc = run_my_logistic_regression(X_train, X_test, y_train, y_test, feature_cols, window_size=ws)
        
        # 儲存當前窗口大小的準確度結果
        results_summary.append({
            "Window_Size": ws,
            "RF_Accuracy": rf_acc,
            "XGB_Accuracy": xgb_acc,
            "My_LR_Accuracy": my_acc
        })


    df_results = pd.DataFrame(results_summary)
    
    print("\n" + "=====================================================================")
    print("模型預測準確度對比表格")
    print("=====================================================================")
    print(df_results.to_string(index=False, formatters={
        'RF_Accuracy': '{:,.4f}'.format,
        'XGB_Accuracy': '{:,.4f}'.format,
        'My_LR_Accuracy': '{:,.4f}'.format
    }))
    print("=====================================================================")
    
    # 找出每個窗口大小下的最佳模型
    print("\n[分析結論]")
    for index, row in df_results.iterrows():
        ws = int(row['Window_Size'])
        accs = {
            "隨機森林": row['RF_Accuracy'],
            "XGBoost": row['XGB_Accuracy'],
            "手寫 Logistic": row['My_LR_Accuracy']
        }
        best_model = max(accs, key=accs.get)
        print(f"窗口天數 {ws:3d} 天的最佳預測模型為: {best_model:<12s} (準確度: {accs[best_model]:.4f})")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

正在下載數據...



[*********************100%***********************]  1 of 1 completed



 啟動資料清洗
資料庫寫入成功。

特徵相關係數與熱點圖分析
[Plot] 熱點圖已成功儲存為 'correlation_heatmap_v2.png'

[Ranking] 各特徵與明日漲跌(Target)的關聯度排名：
-----------------------------------------------------------------
 排名  |      Pearson (線性關聯)       |      Spearman (趨勢關聯)     
-----------------------------------------------------------------
 # 1 | TW_Ret_1D       (-0.0636) | TW_Ret_1D       (-0.0696)
 # 2 | US_GSPC_Ret_5D  (-0.0465) | Close           (-0.0655)
 # 3 | TW_Vol_Ratio_5D (+0.0451) | Low             (-0.0625)
 # 4 | Close           (-0.0437) | High            (-0.0612)
 # 5 | TW_Ret_5D       (-0.0435) | Open            (-0.0604)
 # 6 | Low             (-0.0430) | US_GSPC_Ret_5D  (-0.0570)
 # 7 | High            (-0.0414) | TW_Vol_Ratio_5D (+0.0436)
 # 8 | Open            (-0.0412) | TW_Ret_5D       (-0.0402)
 # 9 | Volume          (+0.0390) | Volume          (+0.0376)
 #10 | US_SOX_Ret_5D   (-0.0302) | US_SOX_Ret_5D   (-0.0320)
 #11 | TW_RSI_14       (-0.0263) | US_SOX_Ret_1D   (-0.0240)
 #12 | TW_BB_Width     